# Demo: Fine-tuning BERT, GPT-2, and T5

## 1. BERT for sentiment classification

- Why BERT?: strong at classification with minimal training
- Task: sentence-level sentiment classification
- Dataset (Hugging Face datasets names): SST-2 (GLUE) – glue

## 2. GPT-2 for dialogue/response generation

- Why GPT?: natural for next-token generation
- Task: short-turn response generation
- Dataset: DailyDialog – daily_dialog

## 3. T5 for dialogue summarization

- Why T5: designed for seq2seq; single text-to-text interface
- Task: summarization (text-to-text)
- Dataset: SAMSum – samsum

In [ ]:
import os, json, random

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling
)

In [ ]:
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Pre-download DailyDialog dataset
raw_data = load_dataset('daily_dialog')
print('DailyDialog done downloading.')

In [ ]:
model_name = 'distilgpt2'
output_dir = 'outputs/distilgpt2-dailydialog'

In [ ]:
# Tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [ ]:
# Tokenize and preprocess data
def to_text(example):
    s = ""
    for u in example['dialog']:
        s += f"<turn> {u}"
    return {'text': s}

data = raw_data.map(to_text)
data

In [ ]:
# Tokenize and preprocess data
max_length = 256

def tokenize(batch):
    enc = tokenizer(batch['text'], truncation=True, max_length=max_length)
    enc['labels'] = enc['input_ids'].copy()
    return enc

tokenized_data = data.map(tokenize, batched=True, remove_columns=data['train'].column_names)

In [ ]:
# Train
args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_data['train'],
    eval_dataset=tokenized_data['validation'],
    data_collator=collator,
    tokenizer=tokenizer
)
trainer.train()

In [ ]:
os.makedirs(output_dir, exist_ok=True)
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
metrics = trainer.evaluate()
if 'eval_loss' in metrics and metrics['eval_loss'] is not None:
    metrics['perplexity'] = float(np.exp(metrics['eval_loss']))
metrics

In [ ]:
with open(os.path.join(output_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved to', output_dir, '', metrics)

In [ ]:
# Quick generation
prompt = """
    <turn> Hi! How are you?
    <turn> I'm good, thanks. What about you?
    <turn>
"""

In [ ]:
model.eval()

In [ ]:
enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)

In [ ]:
with torch.no_grad():
    out = model.generate(
        **enc,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

In [ ]:
prompt

In [ ]:
tokenizer.decode(out[0], skip_special_tokens=True)